# 13 可重現研究 — 參考解答

松柏護理之家退伍軍人症可重現分析流程練習的完整解答。

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

## 題目 1：建立疫情摘要 dict

In [ ]:
path = Path("data/synthetic/legionella_outbreak.csv")
df = pd.read_csv(path)
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

n_infected = int(df["infected"].sum())
n_deaths = int((df["outcome"] == "dead").sum())

summary = {
    "n_residents": len(df),
    "n_infected": n_infected,
    "n_deaths": n_deaths,
    "attack_rate": f"{df['infected'].mean():.1%}",
    "cfr": f"{n_deaths / n_infected:.1%}",
}

print("=== 疫情摘要 ===")
for k, v in summary.items():
    print(f"  {k}: {v}")

print("\n→ 280 位住民、121 人感染、19 人死亡")
print("→ 侵襲率 43.2%、致死率 15.7%")

## 題目 2：可重現檢查清單

In [ ]:
checks = {
    "uv.lock 存在": Path("uv.lock").exists(),
    "pyproject.toml 存在": Path("pyproject.toml").exists(),
    "資料檔存在": Path("data/synthetic/legionella_outbreak.csv").exists(),
}

print("=== 可重現檢查清單 ===")
for item, ok in checks.items():
    status = "✓" if ok else "✗"
    print(f"  [{status}] {item}")

all_pass = all(checks.values())
print(f"\n→ {'全部通過！' if all_pass else '有項目未通過'}")

print("\n=== 每個項目為什麼重要 ===")
print("  uv.lock → 確保所有套件版本一致")
print("  pyproject.toml → 定義專案的套件需求")
print("  資料檔 → 沒有輸入就沒有輸出")

## 題目 3（挑戰題）：摘要輸出與驗證

In [ ]:
import json
import sys

# 存成 CSV
summary_df = pd.DataFrame([summary])
output_path = Path("data/processed")
output_path.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(output_path / "summary.csv", index=False)
print("已存檔：data/processed/summary.csv")

# 重新讀取並驗證
reloaded = pd.read_csv(output_path / "summary.csv")
print(f"\n=== 驗證 ===")
print(f"原始 n_residents: {summary['n_residents']}")
print(f"重讀 n_residents: {reloaded['n_residents'].iloc[0]}")
print(f"一致: {summary['n_residents'] == reloaded['n_residents'].iloc[0]}")

# 版本資訊
print(f"\n=== 環境版本 ===")
print(f"  Python: {sys.version.split()[0]}")
print(f"  pandas: {pd.__version__}")
print(f"  numpy: {np.__version__}")

print("\n=== 可能導致結果不同的因素 ===")
print("  1. 套件版本不同（例如 pandas 行為改變）")
print("  2. Python 版本不同")
print("  3. 資料檔被修改或遺失")
print("  4. 有使用亂數但未固定 seed")
print("  5. 作業系統差異（浮點運算精度）")
print("\n→ 用 uv.lock + git 可以解決前 3 個問題")

### 解讀

- **摘要 dict** 是最小可驗證單位——任何人跑都應該得到 280 住民、121 感染、19 死亡
- **檢查清單** 確保環境完整，缺少任何一項都可能導致無法重現
- **版本記錄** 是除錯的關鍵——如果結果不同，先比對版本
- **可重現三要素**：固定資料 + 版本控制程式碼 + 鎖定環境

## 題目 4：固定隨機種子確保可重現（登革熱情境）

1. 用 `np.random.default_rng(seed)` 以相同種子產生兩次登革熱模擬病例數（各 20 天）
2. 用 `np.array_equal` 驗證兩次結果完全相同
3. 換一個種子再產生一次，說明結果為何不同

In [ ]:
import numpy as np
a = np.random.default_rng(42).poisson(5, 20)
b = np.random.default_rng(42).poisson(5, 20)
print("種子 42 兩次相同：", np.array_equal(a, b))
c = np.random.default_rng(99).poisson(5, 20)
print("種子 99 與 42 相同：", np.array_equal(a, c))
print("解讀：相同種子 → 相同亂數序列 → 結果可重現；不同種子則序列不同。")

## 題目 5：資料檔案雜湊值（COVID-19 情境）

1. 建立一個小的 COVID-19 摘要 DataFrame
2. 用 `hashlib.md5` 對其 CSV 字串算雜湊值
3. 說明為什麼雜湊值可用來驗證資料是否被更動

In [ ]:
import pandas as pd, hashlib
df = pd.DataFrame({"region": ["北", "南", "東", "西"], "cases": [120, 60, 45, 30]})
csv_str = df.to_csv(index=False)
digest = hashlib.md5(csv_str.encode("utf-8")).hexdigest()
print("MD5 =", digest)
print("解讀：資料只要有任何一格改變，雜湊值就會不同 → 可用來驗證資料完整性／是否被竄改。")

## 題目 6：摘要輸出成 JSON 並驗證（麻疹情境）

1. 建立一個麻疹疫情摘要 `dict`（病例、接種率、VE）
2. 用 `json.dump` 存檔，再 `json.load` 讀回
3. 用 `==` 驗證讀回的內容與原始完全相同

In [ ]:
import json
summary = {"cases": 37, "coverage_pct": 88.5, "ve_pct": 93.8}
with open("measles_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False)
with open("measles_summary.json", encoding="utf-8") as f:
    loaded = json.load(f)
print("讀回內容 =", loaded)
print("與原始相同：", loaded == summary)

## 題目 7：可重現的分析函式（諾羅病毒情境）

1. 定義一個有 docstring 的函式 `attack_rate(cases, population)`
2. 函式需驗證輸入（population > 0），並回傳侵襲率
3. 用諾羅病毒宴會數字測試，並印出結果

In [ ]:
def attack_rate(cases, population):
    """侵襲率 = 病例數 / 人口；population 必須 > 0，否則丟出 ValueError。"""
    if population <= 0:
        raise ValueError("population 必須大於 0")
    return cases / population

ar = attack_rate(48, 150)
print(f"諾羅病毒宴會侵襲率 = {ar:.1%}")

## 題目 8（挑戰題）：迷你可重現分析流程（結核情境）

串起完整可重現流程：
1. 固定種子產生合成結核病資料（有 age、smear_positive、cured）
2. 計算痊癒率並存成 JSON
3. 重新讀回 JSON 並驗證數字一致
4. 說明「種子 + 版本 + 輸出雜湊」如何讓分析可被他人重現

In [ ]:
import numpy as np, json
rng = np.random.default_rng(2026)
n = 300
age = rng.integers(18, 85, n)
smear = rng.binomial(1, 0.4, n)
cured = rng.binomial(1, 0.75 - 0.1 * smear, n)
result = {"n": int(n), "cured_rate": round(float(cured.mean()), 4)}
with open("tb_repro.json", "w") as f:
    json.dump(result, f)
with open("tb_repro.json") as f:
    back = json.load(f)
assert back["cured_rate"] == result["cured_rate"]
print("痊癒率 =", result["cured_rate"], "｜可重現流程完成")
print("解讀：固定種子（相同資料）+ 記錄套件版本 + 輸出雜湊/JSON，讓他人能重跑並得到相同數字。")